# BM25 Parameter Sweep Orchestrator

## Purpose
Automate running the full retrieval + evaluation pipeline across multiple BM25 configurations
using Papermill for notebook parameterization.

## Inputs
- Config files in `/work/configs/bm25_unigram__k1-*__b-*.yaml`
- Pre-built indexes in `/work/index/`

## Outputs
- Multiple run directories with metrics for each k1/b combination
- Summary CSV: `/work/runs/bm25_unigram_sweep_summary.csv`

## What This Notebook Does
1. Installs Papermill for notebook parameterization
2. Defines BM25 parameter grid (k1: 0.6-1.6, b: 0.2-0.95)
3. For each parameter combination:
   - Runs `index_builder.ipynb` with config
   - Runs `retrieve.ipynb` to get candidates
   - Runs `metrics.ipynb` to compute evaluation metrics
4. Aggregates results into summary table
5. Identifies best-performing parameter combination

In [1]:
!pip install papermill

In [2]:
# --- MODE: choose "uni" for BM25 unigrams, "uni2" for BM25 1+2-grams, "params" --- for unigrams + parameters
MODE = "uni2"   # "uni", "uni2" or "params"

In [3]:
# Setup & sanity checks
# %% [markdown]
# # BM25 (k1, b) sweep — orchestrator (uses your existing notebooks)
# - Generates one YAML per (k1, b)
# - Runs: index_build.ipynb → retrieve.ipynb → metrics.ipynb (via Papermill)
# - Writes standard outputs into /work/index/<METHOD_NAME>/ and /work/runs/<METHOD_NAME>/
# - Stores executed notebook logs in /work/logs/

# %% 
from __future__ import annotations
import os, json, shutil
from pathlib import Path

try:
    import papermill as pm
except ImportError as e:
    raise SystemExit(
        "Papermill is not installed in this environment.\n"
        "Install it with:  pip install papermill  (or)  conda install -c conda-forge papermill"
    )

os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

# ---- Paths (adjust if your repo layout differs) ----
WORK       = Path("/work")
SRC        = WORK / "src"
NOTEBOOKS  = WORK / "notebooks"
CONFIGS    = WORK / "configs"
INDEX_ROOT = WORK / "index"
RUNS       = WORK / "runs"
LOGS       = WORK / "logs"
DATA_DIR   = WORK / "data" / "processed"

NB_BUILD    = SRC / "index_builder.ipynb"
NB_RETRIEVE = SRC / "retrieve.ipynb"
NB_METRICS  = SRC / "metrics.ipynb"

# ensure src is on Python path for child notebooks
os.environ["PYTHONPATH"] = f"{SRC}:{os.environ.get('PYTHONPATH','')}"
PM_CWD = str(WORK)   # run all notebooks with /work as cwd

for p in [CONFIGS, INDEX_ROOT, RUNS, LOGS]:
    p.mkdir(parents=True, exist_ok=True)

# ---- Data sources (must exist) ----
COLLECTION = "OEB"
QUERIES_PATH       = DATA_DIR / f"{COLLECTION}_short_feats.parquet"
CORPUS_PATH        = DATA_DIR / f"{COLLECTION}_long_feats.parquet"
FEATURES_META_PATH = DATA_DIR / f"{COLLECTION}_features_meta.json"

missing = [str(p) for p in [NB_BUILD, NB_RETRIEVE, NB_METRICS, QUERIES_PATH, CORPUS_PATH, FEATURES_META_PATH] if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required paths:\n- " + "\n- ".join(missing))

print("Environment OK.\n",
      f"Notebooks: {NB_BUILD.name}, {NB_RETRIEVE.name}, {NB_METRICS.name}\n",
      f"Data: {QUERIES_PATH.name}, {CORPUS_PATH.name}, {FEATURES_META_PATH.name}")


Environment OK.
 Notebooks: index_builder.ipynb, retrieve.ipynb, metrics.ipynb
 Data: OEB_short_feats.parquet, OEB_long_feats.parquet, OEB_features_meta.json


In [4]:
# Grid & method naming
# %%
if MODE == "uni":
    NGRAM_RANGE = [1, 1]
    TAG = "bm25_unigram"
    # Coarse dev sweep tuned for your corpus (unigrams)
    K1S = [0.6, 0.8, 1.0, 1.2, 1.4]
    BS  = [0.20, 0.35, 0.50, 0.65, 0.80]
elif MODE == "uni2":
    NGRAM_RANGE = [1, 2]
    TAG = "bm25_unibigram"
    # Slightly shifted grid for bigrams
    K1S = [0.8, 1.0, 1.2, 1.4, 1.6]
    BS  = [0.35, 0.50, 0.65, 0.80, 0.95]
elif MODE == "params":                
    NGRAM_RANGE = [1, 1]
    TAG = "bm25_unigram_params"
    K1S = [0.6, 0.8, 1.0, 1.2]         
    BS  = [0.35, 0.50, 0.65]
else:
    raise ValueError("MODE must be 'uni' or 'uni2' or 'params'")

# Retrieval settings
TOP_K        = 100
DEV_SAMPLE   = 16590   # number of queries for dev selection
FULL_EVAL    = False  # if True, runs the full set instead of the dev sample for every variant

# Notebook parameters you can tweak later from here too:
DEFAULT_RETRIEVE_PARAMS = dict(
    SAVE=True,
    K=TOP_K,
)

def method_name(k1: float, b: float) -> str:
    return f"{TAG}__k1-{k1:.2f}__b-{b:.2f}"


In [5]:
# YAML writer (per (k₁, b))
# %%
def write_yaml_for(method: str, k1: float, b: float) -> Path:
    """
    Writes a JSON-as-YAML-equivalent config. Your index_build notebook just needs a dict file path.
    """
    cfg = {
        "collection": COLLECTION,
        "paths": {"data_dir": str(DATA_DIR), "index_root": str(INDEX_ROOT)},
        "inputs": {
            "short_feats": str(QUERIES_PATH),
            "long_feats":  str(CORPUS_PATH),
            "features_meta": str(FEATURES_META_PATH),
        },
        "method": {
            "family": "bm25",
            "name": method,
            "impl": TAG,
            "save_as": method,
            "params": {
                "analyzer": "word",
                "ngram_range": NGRAM_RANGE,
                "lowercase": True,
                "token_pattern": r"(?u)\b\w+\b",
                "strip_accents": "unicode",
                "k1": float(k1),
                "b": float(b),
            },
        },
        "io": {
            "data_dirname": "data",
            "mapping_file": "mapping.jsonl",
            "fields_file": "fields.json",
            "meta_file": "meta.json",
        },
    }
    path = CONFIGS / f"{method}.yaml"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=2)
    return path


In [6]:
# One-variant runner (build → retrieve → metrics)
# %%
def run_variant(k1: float, b: float, dev_mode: bool = True):
    """
    Calls your notebooks via Papermill with parameters injected.
    dev_mode=True → retrieve on a random subset (DEV_SAMPLE queries)
    dev_mode=False → full set
    """
    m = method_name(k1, b)
    cfg_path = write_yaml_for(m, k1, b)
    run_dir  = RUNS / m
    run_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n=== {m} ===")
    print(f" \n== {TAG} ==")

    # 1) Build index (writes /work/index/<m> and updates registry.json)
    pm.execute_notebook(
        input_path=str(NB_BUILD),
        output_path=str(LOGS / f"build__{TAG}.ipynb"),
        parameters={"CFG_PATH": str(cfg_path),
                   "RUN_CONTEXT": "orchestrator",},
        cwd=PM_CWD,
    )

    # 2) Retrieve (dev subset or full), writes results_topK.jsonl.gz in /work/runs/<m>/
    retrieve_params = dict(DEFAULT_RETRIEVE_PARAMS)
    retrieve_params.update({
        "METHOD_NAME": m,
        "RANDOM_SAMPLE": (None if not dev_mode else int(DEV_SAMPLE)),
        "K": 100,
        "SAVE": True,
        "RUN_CONTEXT": "orchestrator",
    })
    pm.execute_notebook(
        input_path=str(NB_RETRIEVE),
        output_path=str(LOGS / f"retrieve__{TAG}.ipynb"),
        parameters=retrieve_params,
        cwd=PM_CWD,
    )

    # 3) Metrics (reads the latest results_topK.jsonl.gz; writes metrics_* into /work/runs/<m>/)
    pm.execute_notebook(
        input_path=str(NB_METRICS),
        output_path=str(LOGS / f"metrics__{TAG}.ipynb"),
        parameters={"RUN_DIR": str(run_dir),
                   "RUN_CONTEXT": "orchestrator",},
        cwd=PM_CWD,
    )

    print(f"[OK] Finished {m} → see {run_dir} for metrics and {LOGS} for executed notebooks.")


In [7]:
# Sweep runner + summary table
# %%
import pandas as pd

def run_sweep(k1s=K1S, bs=BS, full_eval: bool = FULL_EVAL):
    """
    Loops the grid and runs each variant.
    Produces a summary CSV with Acc@1 (item) and MRR from metrics_dual.csv of each run.
    """
    summary_rows = []
    for k1 in k1s:
        for b in bs:
            run_variant(k1, b, dev_mode=(not full_eval))

            # read the just-written metrics_dual.csv and collect a couple of headline metrics
            m = method_name(k1, b)
            metrics_csv = RUNS / m / "metrics_dual.csv"
            if metrics_csv.exists():
                df = pd.read_csv(metrics_csv)
                row_item_overall = df[(df["target"]=="item") & (df["scope"]=="overall")]
                if not row_item_overall.empty:
                    r = row_item_overall.iloc[0].to_dict()
                    summary_rows.append({
                        "method": m,
                        "k1": k1, "b": b,
                        "queries": r.get("queries"),
                        "Acc@1_item": r.get("Acc@1"),
                        "Recall@5_item": r.get("Recall@5"),
                        "Recall@10_item": r.get("Recall@10"),
                        "MRR_item": r.get("MRR"),
                        "nDCG@10_item": r.get("nDCG@10"),
                    })
            else:
                print(f"[WARN] Missing metrics for {m}: {metrics_csv}")

    if summary_rows:
        df_sum = pd.DataFrame(summary_rows).sort_values(
            ["Acc@1_item","MRR_item"], ascending=[False, False]
        )
        out_csv = RUNS / f"{TAG}_sweep_summary.csv"
        df_sum.to_csv(out_csv, index=False)
        display(df_sum.head(10))
        print(f"\n[SUMMARY] Wrote {out_csv}")
    else:
        print("[SUMMARY] No rows collected — check earlier logs.")

# Kick it off 🚀
run_sweep()



=== bm25_unibigram__k1-0.80__b-0.35 ===
 
== bm25_unibigram ==


Executing:   0%|          | 0/5 [00:00<?, ?cell/s]

Executing:   0%|          | 0/5 [00:00<?, ?cell/s]

Executing:   0%|          | 0/16 [00:00<?, ?cell/s]

[OK] Finished bm25_unibigram__k1-0.80__b-0.35 → see /work/runs/bm25_unibigram__k1-0.80__b-0.35 for metrics and /work/logs for executed notebooks.

=== bm25_unibigram__k1-0.80__b-0.50 ===
 
== bm25_unibigram ==


...

Executing:   0%|          | 0/5 [00:00<?, ?cell/s]

Executing:   0%|          | 0/16 [00:00<?, ?cell/s]

[OK] Finished bm25_unibigram__k1-1.60__b-0.95 → see /work/runs/bm25_unibigram__k1-1.60__b-0.95 for metrics and /work/logs for executed notebooks.


,method,k1,b,queries,Acc@1_item,Recall@5_item,Recall@10_item,MRR_item,nDCG@10_item
0,bm25_unibigram__k1-0.80__b-0.35,0.8,0.35,16590.0,0.826703,0.868596,0.869681,0.844137,0.849523
5,bm25_unibigram__k1-1.00__b-0.35,1.0,0.35,16590.0,0.824834,0.867932,0.869620,0.842583,0.848308
10,bm25_unibigram__k1-1.20__b-0.35,1.2,0.35,16590.0,0.823689,0.866606,0.869379,0.841490,0.847372
15,bm25_unibigram__k1-1.40__b-0.35,1.4,0.35,16590.0,0.821278,0.865280,0.869259,0.839768,0.846006
1,bm25_unibigram__k1-0.80__b-0.50,0.8,0.50,16590.0,0.820856,0.864436,0.868113,0.840229,0.846114
20,bm25_unibigram__k1-1.60__b-0.35,1.6,0.35,16590.0,0.820675,0.864135,0.869078,0.839196,0.845485
6,bm25_unibigram__k1-1.00__b-0.50,1.0,0.50,16590.0,0.818143,0.862025,0.866546,0.837879,0.843869
2,bm25_unibigram__k1-0.80__b-0.65,0.8,0.65,16590.0,0.816757,0.860579,0.864678,0.836879,0.842580
11,bm25_unibigram__k1-1.20__b-0.50,1.2,0.50,16590.0,0.816335,0.861181,0.865521,0.836387,0.842434
16,bm25_unibigram__k1-1.40__b-0.50,1.4,0.50,16590.0,0.814225,0.859795,0.864256,0.834646,0.840734



[SUMMARY] Wrote /work/runs/bm25_unibigram_sweep_summary.csv
